## Scraping a dedicated page : Under the Queen's Umbrella, Research Question : Comparing overall reviews to story, acting/cast, music, and rewatch value reviews

## Setting up the get request 

In [1]:
import requests
import time
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
URL = "https://mydramalist.com/705857-umbrella/reviews"
page = requests.get(URL, headers=headers, timeout=10)
page.raise_for_status()
#Checking status code
print(page.status_code)
print(page.text[:500])

200
<!DOCTYPE html>
<html lang="en-US">
<head>
<title>Under the Queen's Umbrella Reviews - MyDramaList</title>
<link rel="dns-prefetch" href="//fonts.gstatic.com">
<link rel="dns-prefetch" href="//fonts.googleapis.com">
<link rel="dns-prefetch" href="//apis.google.com">
<link rel="dns-prefetch" href="//tpc.googlesyndication.com">
<link rel="dns-prefetch" href="//googleads.g.doubleclick.net">
<link rel="dns-prefetch" href="//securepubads.g.doubleclick.net">
<link rel="preconnect" href="//fonts.gstati


In [4]:
def get_reviews(response):
    soup = BeautifulSoup(response.content, "html.parser")
    reviews = []
    container_class = "review"

    containers = soup.find_all("div", class_=container_class)

    for item in containers:
        #name 
        name_tag = None
        for link in item.find_all("a"):
            if link.has_attr("href") and "/profile/" in link["href"]:
                after_profile = link["href"].split("/profile/")[1]
                if "/" not in after_profile and link.text.strip() != "":
                    name_tag = link
                    break

        #overall
        overall_wrapper = item.find(class_="rating-overall")
        overall_tag = overall_wrapper.find("span", class_="score") if overall_wrapper else None

        #story, acting/cast, music, rewatch value 
        ratings_wrapper = item.find(class_="review-rating")
        individual_ratings = ratings_wrapper.find_all("div") if ratings_wrapper else []

        ratings = {}
        for individual_rating in individual_ratings:
            parts = individual_rating.text.strip().split()     
            label = " ".join(parts[:-1])                    
            ratings[label] = float(parts[-1])               

        name = name_tag.text.strip() if name_tag else None
        profile = name_tag["href"] if name_tag else None
        overall = float(overall_tag.text.strip()) if overall_tag else None

        reviews.append({
            "name": name,
            "profile": profile,
            "overall": overall,
            "story": ratings.get("Story"),
            "acting": ratings.get("Acting/Cast"),
            "music": ratings.get("Music"),
            "rewatch": ratings.get("Rewatch Value"),
        })

    return reviews


In [5]:
if __name__ == "__main__":
    all_reviews = []
#only did 3 pages but there are more 
    for n in range(1, 3):
        response = requests.get(f"{URL}?page={n}", headers=headers, timeout=10)
        #showing status 
        print(f"Page {n}: status {response.status_code}")
        page_reviews = get_reviews(response)
        if not page_reviews:
            break
        all_reviews.extend(page_reviews)
        time.sleep(2)

    for review in all_reviews[:23]:
        print(review)

Page 1: status 200
Page 2: status 200
{'name': 'nyx', 'profile': '/profile/10052789', 'overall': 10.0, 'story': 10.0, 'acting': 10.0, 'music': 10.0, 'rewatch': 10.0}
{'name': 'Salwa Nice', 'profile': '/profile/7656325', 'overall': 9.0, 'story': 9.5, 'acting': 9.0, 'music': 8.0, 'rewatch': 8.0}
{'name': 'seltzer', 'profile': '/profile/ryoato', 'overall': 9.0, 'story': 8.5, 'acting': 9.5, 'music': 10.0, 'rewatch': 8.0}
{'name': 'Mash-mallow', 'profile': '/profile/Marshiie', 'overall': 9.5, 'story': 9.5, 'acting': 10.0, 'music': 9.5, 'rewatch': 9.0}
{'name': 'cejj', 'profile': '/profile/timmyrate', 'overall': 9.5, 'story': 9.0, 'acting': 10.0, 'music': 10.0, 'rewatch': 9.0}
{'name': 'mayra', 'profile': '/profile/mayrasgalaxy', 'overall': 10.0, 'story': 10.0, 'acting': 10.0, 'music': 10.0, 'rewatch': 10.0}
{'name': 'Kate', 'profile': '/profile/theKate', 'overall': 8.0, 'story': 8.0, 'acting': 8.5, 'music': 7.5, 'rewatch': 7.0}
{'name': 'Salatheel', 'profile': '/profile/Salatheel', 'overall